1. Import Neccessary Libraries 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set professional viz style
plt.style.use('fivethirtyeight')
sns.set_theme(style="whitegrid")

print("Nova Financial Quantitative Environment: READY")

Nova Financial Quantitative Environment: READY


2. Data Loading Stock Price Dataset 

2. Load the Stock Price Dataset

In [4]:
# =================================================================
# TASK 2: DATA PREPARATION PIPELINE (Multi-Ticker)
# =================================================================
# Configuration based on your folder structure
data_folder = '../data/raw/yfinance_data/Data/'
tickers = ['AAPL', 'AMZN', 'GOOG', 'META', 'NVDA']
stock_dict = {}

print("--- STARTING RIGOROUS DATA PREPARATION ---")

for ticker in tickers:
    file_path = os.path.join(data_folder, f"{ticker}.csv")
    
    if os.path.exists(file_path):
        # ---------------------------------------------------------
        # 1. LOAD THE STOCK PRICE DATASET
        # ---------------------------------------------------------
        df = pd.read_csv(file_path)
        
        # Standardizing headers (Removes hidden spaces that cause KeyErrors)
        df.columns = [c.strip() for c in df.columns]
        
        # ---------------------------------------------------------
        # 2. ENSURE COLUMNS ARE CORRECTLY TYPED
        # ---------------------------------------------------------
        # Convert Date to datetime
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'])
            # Set Date as index for financial time-series analysis
            df.set_index('Date', inplace=True)
        
        # Convert Price/Volume columns to numeric (Floats/Ints)
        # We include 'Adj Close' as it is best practice for return calculations
        target_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
        for col in target_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # ---------------------------------------------------------
        # 3. CHECK FOR AND HANDLE MISSING VALUES
        # ---------------------------------------------------------
        initial_missing = df.isnull().sum().sum()
        
        if initial_missing > 0:
            # Handle using Forward Fill (Financial Industry Standard)
            df.ffill(inplace=True)
            # Handle any remaining at the start using Backward Fill
            df.bfill(inplace=True)
            status = f"Fixed {initial_missing} missing values"
        else:
            status = "No missing values detected"
            
        # Store cleaned data
        stock_dict[ticker] = df
        print(f"✅ {ticker:4} | {status} | Shape: {df.shape}")
        
    else:
        print(f"❌ {ticker:4} | File not found at {file_path}")

print("\n--- DATA PREPARATION COMPLETE ---")

--- STARTING RIGOROUS DATA PREPARATION ---
✅ AAPL | No missing values detected | Shape: (3774, 5)
✅ AMZN | No missing values detected | Shape: (3774, 5)
✅ GOOG | No missing values detected | Shape: (3774, 5)
✅ META | No missing values detected | Shape: (2923, 5)
✅ NVDA | No missing values detected | Shape: (3774, 5)

--- DATA PREPARATION COMPLETE ---


3. Verification & Portfolio Audit

In [7]:
# =================================================================
# TASK: COMPUTE TECHNICAL INDICATORS WITH TA-LIB
# =================================================================
import talib

print("--- COMPUTING & ANALYZING TECHNICAL INDICATORS ---")

for ticker, df in stock_dict.items():
    # ---------------------------------------------------------
    # 1. MOVING AVERAGES (MA): Calculate SMA and EMA over multiple windows
    # ---------------------------------------------------------
    # Short-term (20-day) and Medium-term (50-day) windows
    df['SMA_20'] = talib.SMA(df['Close'], timeperiod=20)
    df['SMA_50'] = talib.SMA(df['Close'], timeperiod=50)
    
    df['EMA_20'] = talib.EMA(df['Close'], timeperiod=20)
    df['EMA_50'] = talib.EMA(df['Close'], timeperiod=50)

    # ---------------------------------------------------------
    # 2. RSI: Identify overbought and oversold conditions
    # ---------------------------------------------------------
    df['RSI'] = talib.RSI(df['Close'], timeperiod=14)
    
    # Explicitly identifying the conditions
    df['RSI_Condition'] = 'Neutral'
    df.loc[df['RSI'] > 70, 'RSI_Condition'] = 'Overbought'
    df.loc[df['RSI'] < 30, 'RSI_Condition'] = 'Oversold'

    # ---------------------------------------------------------
    # 3. MACD: Detect momentum shifts and potential trend reversals
    # ---------------------------------------------------------
    macd, macdsignal, macdhist = talib.MACD(df['Close'], 
                                            fastperiod=12, 
                                            slowperiod=26, 
                                            signalperiod=9)
    df['MACD'] = macd
    df['MACD_Signal'] = macdsignal
    df['MACD_Hist'] = macdhist
    
    # Detecting Momentum Shift: MACD crossing the Signal line
    # Bullish Shift = MACD > Signal; Bearish Shift = MACD < Signal
    df['MACD_Sentiment'] = np.where(df['MACD'] > df['MACD_Signal'], 'Bullish', 'Bearish')

    print(f"✅ {ticker:4} | MAs, RSI Conditions, and MACD Shifts Processed.")

print("\n--- ALL TASKS COMPLETE ---")

# Display proof of 'Identification' and 'Detection' for AAPL
display(stock_dict['AAPL'][['Close', 'RSI', 'RSI_Condition', 'MACD_Sentiment']].tail(10))

--- COMPUTING & ANALYZING TECHNICAL INDICATORS ---
✅ AAPL | MAs, RSI Conditions, and MACD Shifts Processed.
✅ AMZN | MAs, RSI Conditions, and MACD Shifts Processed.
✅ GOOG | MAs, RSI Conditions, and MACD Shifts Processed.
✅ META | MAs, RSI Conditions, and MACD Shifts Processed.
✅ NVDA | MAs, RSI Conditions, and MACD Shifts Processed.

--- ALL TASKS COMPLETE ---


,Close,RSI,RSI_Condition,MACD_Sentiment
Date,,,,
2023-12-15,195.721619,67.991716,Neutral,Bullish
2023-12-18,194.057343,62.680148,Neutral,Bullish
2023-12-19,195.097504,64.544428,Neutral,Bearish
2023-12-20,193.007248,58.247457,Neutral,Bearish
2023-12-21,192.858643,57.815603,Neutral,Bearish
2023-12-22,191.788757,54.672784,Neutral,Bearish
2023-12-26,191.243912,53.090049,Neutral,Bearish
2023-12-27,191.342972,53.354446,Neutral,Bearish
2023-12-28,191.768951,54.540999,Neutral,Bearish


4. Compute Financial Metrics (Returns & Volatility)

In [17]:
# =================================================================
# TASK: APPLY PYNANCE METRICS SUITE (Resilient Implementation)
# Goal: Compute Returns and Volatility while handling column variations.
# =================================================================
import numpy as np

print("--- COMPUTING RESILIENT QUANTITATIVE METRICS ---")

for ticker, df in stock_dict.items():
    # 1. SAFE COLUMN DETECTION
    # Professional datasets vary: we prioritize 'Adj Close', fall back to 'Close'
    if 'Adj Close' in df.columns:
        target_col = 'Adj Close'
    elif 'Close' in df.columns:
        target_col = 'Close'
    else:
        print(f"❌ Error for {ticker}: No Close column found. Skipping.")
        continue

    # 2. PYNANCE METRIC: Arithmetic Daily Returns
    # Using the detected 'target_col' to avoid KeyErrors
    df['Daily_Return'] = df[target_col].pct_change()

    # 3. PYNANCE METRIC: Logarithmic Returns
    # 'Additional Setting' for statistical modeling
    df['Log_Return'] = np.log(df[target_col] / df[target_col].shift(1))

    # 4. PYNANCE METRIC: Annualized Volatility
    # 21-day rolling window annualized
    df['Volatility'] = df['Daily_Return'].rolling(window=21).std() * np.sqrt(252)

    # 5. PYNANCE METRIC: Cumulative Growth
    df['Cumulative_Return'] = (1 + df['Daily_Return']).cumprod() - 1

    print(f"✅ Metrics computed for {ticker:4} using column: [{target_col}]")

print("\n--- ALL ADDITIONAL METRICS COMPLETE ---")
# Preview results for a known ticker
if 'AAPL' in stock_dict:
    display(stock_dict['AAPL'][[target_col, 'Daily_Return', 'Volatility']].tail())

--- COMPUTING RESILIENT QUANTITATIVE METRICS ---
✅ Metrics computed for AAPL using column: [Close]
✅ Metrics computed for AMZN using column: [Close]
✅ Metrics computed for GOOG using column: [Close]
✅ Metrics computed for META using column: [Close]
✅ Metrics computed for NVDA using column: [Close]

--- ALL ADDITIONAL METRICS COMPLETE ---


,Close,Daily_Return,Volatility
Date,,,
2023-12-22,191.788757,-0.005548,0.143062
2023-12-26,191.243912,-0.002841,0.140983
2023-12-27,191.342972,0.000518,0.140845
2023-12-28,191.768951,0.002226,0.140680
2023-12-29,190.728775,-0.005424,0.140688
